# Deepfake Detection — GPU Retraining Notebook (Phase 1 parallel workstream)

Retrains the Xception-LSTM detector with the **corrected pipeline**:

1. **Leakage-controlled split** — uses `training/dataset_manifest.json` produced by
   `training/make_identity_split.py` (original/manipulated pairs never cross splits).
2. **Correct Xception normalization** — inputs scaled to `[-1, 1]`
   (`keras.applications.xception.preprocess_input` convention) instead of the legacy `[0, 1]`.
3. **Metric-based checkpointing** — best checkpoint selected by validation ROC-AUC (configurable),
   with early stopping; not just `val_loss` on a 13-video set.

## What you need in Google Drive (set once in the mount cell)

```
MyDrive/deepfake_fyp/
├── dataset_manifest.json          # from training/make_identity_split.py (committed in repo)
├── frames.zip                     # processed_dataset/ frames zipped, so it contains
│                                  #   train/<REAL|FAKE>/<video>/frame_*.jpg, val/..., etc.
└── runs/                          # outputs (checkpoints, manifests, evaluations) land here
```

If you only have raw videos, run `python preprocessing.py --mode dataset` locally first.


In [ ]:
#@title 1) GPU check
import tensorflow as tf
print('TensorFlow:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPUs:', gpus)
assert gpus, 'No GPU runtime: use Runtime > Change runtime type > GPU (Colab).'


In [ ]:
#@title 2) Get the repository and install dependencies
REPO_URL = 'https://github.com/TalhaQureshi324/Explainable-DeepFake-detection-by-Xception-LSTM-and-LIME.git'
BRANCH = 'phase-1-backend'   # merge to main after review; update if needed
!rm -rf fyp_repo
!git clone --branch {BRANCH} {REPO_URL} fyp_repo
%cd fyp_repo
!pip -q install -r requirements.txt


In [ ]:
#@title 3) Mount Drive and unpack frames
from google.colab import drive
drive.mount('/content/drive')

import zipfile, pathlib
DRIVE_DIR = pathlib.Path('/content/drive/MyDrive/deepfake_fyp')  # <- edit if different
FRAMES_ZIP = DRIVE_DIR / 'frames.zip'
FRAMES_ROOT = pathlib.Path('/content/frames')

assert FRAMES_ZIP.exists(), f'Missing {FRAMES_ZIP} — upload frames.zip to Drive first.'
if not FRAMES_ROOT.exists():
    print('Unzipping frames (one-time, a few minutes)...')
    FRAMES_ROOT.mkdir(parents=True)
    with zipfile.ZipFile(FRAMES_ZIP) as z:
        z.extractall(FRAMES_ROOT)
# frames.zip may contain processed_dataset/ at the top level; normalize the root:
if (FRAMES_ROOT / 'processed_dataset').is_dir():
    FRAMES_ROOT = FRAMES_ROOT / 'processed_dataset'
print('frames root:', FRAMES_ROOT)
print('manifest   :', DRIVE_DIR / 'dataset_manifest.json')


In [ ]:
#@title 4) Sanity-check the split manifest against the frame folders
import json, pathlib
manifest = json.load(open(DRIVE_DIR / 'dataset_manifest.json'))
counts = {}
missing = 0
frames_root = pathlib.Path(FRAMES_ROOT)
index = {}
for split in ('train','val','calibration','test'):
    for label in ('REAL','FAKE'):
        d = frames_root / split / label
        if d.is_dir():
            for f in d.iterdir():
                if f.is_dir(): index[f.name] = f
for rec in manifest['videos']:
    counts[rec['assigned_split']] = counts.get(rec['assigned_split'], 0) + 1
    if rec['video'] not in index:
        missing += 1
print('videos per split:', counts)
print('videos without preprocessed frames:', missing, '(these are skipped by the trainer)')
assert counts.get('train', 0) > 0 and counts.get('val', 0) > 0


In [ ]:
#@title 5) Configure the experiment
CONFIG = {
    'experiment_name': 'exp01_xception_lstm_corrected',
    'num_frames': 32,            # 32 = V2-style; use 10 + --v1-first-k 10 for V1-style
    'preprocessing_mode': 'minus_one_plus_one',   # ImageNet-correct; zero_one = legacy comparison
    'backbone': 'finetune',      # 'frozen' for a cheaper ablation
    'epochs': 15,
    'batch_size': 8,             # reduce to 4 if the GPU runs out of memory
    'lr': 1e-4,
    'selection_metric': 'val_auc_curve',
    'patience': 4,
    'seed': 42,
}
print(CONFIG)


In [ ]:
!python training/train_corrected.py \
    --manifest "{DRIVE_DIR / 'dataset_manifest.json'}" \
    --frames-root "{FRAMES_ROOT}" \
    --out-dir "training/runs/{CONFIG['experiment_name']}" \
    --num-frames {CONFIG['num_frames']} \
    --preprocessing-mode {CONFIG['preprocessing_mode']} \
    --backbone {CONFIG['backbone']} \
    --epochs {CONFIG['epochs']} \
    --batch-size {CONFIG['batch_size']} \
    --lr {CONFIG['lr']} \
    --selection-metric {CONFIG['selection_metric']} \
    --patience {CONFIG['patience']} \
    --seed {CONFIG['seed']}

In [ ]:
#@title 7) Training curves
import pandas as pd, matplotlib.pyplot as plt, pathlib
log_csv = pathlib.Path(f"training/runs/{CONFIG['experiment_name']}/training_log.csv")
df = pd.read_csv(log_csv)
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
for col, a in (('loss', ax[0]), ('val_auc_curve', ax[1]), ('recall', ax[2])):
    for c in ([col, f'val_{col}'] if col != 'val_auc_curve' else ['val_auc_curve']):
        if c in df: a.plot(df[c], label=c)
    a.set_title(col); a.legend()
plt.tight_layout(); plt.show()


In [ ]:
!python evaluation/evaluate_frames.py \
    --model "training/runs/{CONFIG['experiment_name']}/best_checkpoint.keras" \
    --mode v2 --split val \
    --data-dir "{FRAMES_ROOT}" \
    --out "evaluation/results_exp01_val.json" 

In [ ]:
!python evaluation/evaluate_frames.py \
    --model "training/runs/{CONFIG['experiment_name']}/best_checkpoint.keras" \
    --mode v2 --split calibration \
    --data-dir "{FRAMES_ROOT}" \
    --out "evaluation/results_exp01_calibration.json" 

In [ ]:
#@title 8b) Threshold calibration on the CALIBRATION split only (never test)
!python evaluation/threshold_analysis.py


In [ ]:
#@title 9) Export artifacts to Drive
import shutil, pathlib
run_dir = pathlib.Path(f"training/runs/{CONFIG['experiment_name']}")
export = DRIVE_DIR / 'runs' / CONFIG['experiment_name']
export.mkdir(parents=True, exist_ok=True)
shutil.copy(run_dir / 'best_checkpoint.keras', export / 'best_checkpoint.keras')
shutil.copy(run_dir / 'experiment_manifest.json', export / 'experiment_manifest.json')
shutil.copy(run_dir / 'training_log.csv', export / 'training_log.csv')
for f in ('evaluation/results_exp01_val.json', 'evaluation/results_exp01_calibration.json'):
    if pathlib.Path(f).exists():
        shutil.copy(f, export / pathlib.Path(f).name)
print('exported to', export)


## After training

1. Download `best_checkpoint.keras` + `experiment_manifest.json` from Drive.
2. Put the checkpoint into the backend's `DEEPFAKE_MODELS_DIR` and register it in
   `backend/app/services/model_registry.py` (one entry) — **do not overwrite the historical
   V1/V2 files**; the registry supports side-by-side versions.
3. Pick the decision threshold on the **calibration** split (never on test), set the
   model-specific env override (e.g. `DEEPFAKE_THRESHOLD_V3`), then run the LOCKED
   test split exactly once via `evaluation/evaluate_frames.py --split test` for the report.

### Kaggle notes
- Enable GPU acceleration (Settings → Accelerator → GPU T4/P100).
- Skip the Drive mount cell; attach the dataset via `../input/...` paths and set
  `DRIVE_DIR` / `FRAMES_ROOT` accordingly.
- Everything else (training CLI, evaluation scripts) is identical.
